**Evaluation: 04_ragas_evaluation**
- Runs RAGAS evaluation on expert Q&A pairs across all 5 retrieval strategies.

**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
os.environ["USE_ADLS_GRAPH"] = "true"  # PathRAG uses in-memory graph from ADLS, not Cosmos Gremlin
print("Environment variables set.")

In [ ]:
# Gold/Silver paths
MODE = os.getenv("PIPELINE_MODE", "llm")
LOCAL_GOLD = f"/tmp/pipeline/gold_{MODE}"
LOCAL_SILVER = f"/tmp/pipeline/silver_{MODE}"

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**Download graph files and entity catalog from ADLS**
- PathRAG needs the full graph in memory (nodes.json + edges.json) for DFS path finding.
- Entity catalog is needed for alias expansion at query time.

In [ ]:
import os
from src.storage.adls_adapter import ADLSAdapter

adls = ADLSAdapter()
MODE = os.getenv("PIPELINE_MODE", "llm")
LOCAL_GOLD = f"/tmp/pipeline/gold_{MODE}"

# Download graph files for PathRAG in-memory DFS
kg_dir = f"{LOCAL_GOLD}/knowledge_graph"
os.makedirs(kg_dir, exist_ok=True)
for fname in ["nodes.json", "edges.json"]:
    adls_path = f"gold_{MODE}/knowledge_graph/{fname}"
    data = adls.read_bytes(adls_path)
    if data:
        with open(f"{kg_dir}/{fname}", "wb") as f:
            f.write(data)
        print(f"Downloaded {fname} ({len(data)} bytes)")
    else:
        print(f"WARNING: {adls_path} not found on ADLS")

# Download entity catalog for alias expansion
catalog_data = adls.read_bytes(f"gold_{MODE}/entity_catalog.json")
if catalog_data:
    with open(f"{LOCAL_GOLD}/entity_catalog.json", "wb") as f:
        f.write(catalog_data)
    print(f"Downloaded entity_catalog.json ({len(catalog_data)} bytes)")
else:
    print("WARNING: entity_catalog.json not found on ADLS")

**Configuration**

In [ ]:
from datetime import datetime

# --- CONFIGURE THESE FOR EACH RUN ---
RUN_NAME = "v1_full_dataset"  # Change for each benchmark run
OUTPUT_FILE = f"ragas_{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
ADLS_OUTPUT_PATH = f"evaluation/{OUTPUT_FILE}"

# Which strategies to evaluate
#STRATEGIES_TO_EVAL = ["vector", "pathrag", "graphrag", "hybrid", "react"]
STRATEGIES_TO_EVAL = ["vector"]

print(f"Run name: {RUN_NAME}")
print(f"Output: {OUTPUT_FILE}")
print(f"ADLS: {ADLS_OUTPUT_PATH}")
print(f"Strategies: {STRATEGIES_TO_EVAL}")


**Expert Q&A pairs**

In [ ]:
# Expert question / ground-truth pairs for RAGAS.
# Replace these illustrative examples with questions written by domain experts
# for your own corpus, or load them from a JSON file on ADLS:
#   EXPERT_QA = json.loads(adls.read_bytes("evaluation/expert_qa.json"))
EXPERT_QA = [
    {
        "question": "When did the team complete the migration of the ERP system to the cloud platform?",
        "ground_truth": "March 2024",
    },
    {
        "question": "What availability target was agreed in the service-level agreement for the new data centre?",
        "ground_truth": "99.95% monthly uptime",
    },
    {
        "question": "What did the project lead recommend to reduce failed deployments in Project Atlas?",
        "ground_truth": "Add an automated canary stage before each production rollout",
    },
]

print(f"Expert Q&A pairs: {len(EXPERT_QA)}")
for i, qa in enumerate(EXPERT_QA):
    print(f"  Q{i+1}: {qa['question'][:60]}")

**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Initialize retriever**

In [ ]:
from src.retrieval import HybridRetriever, RetrievalStrategy
from src.storage.cosmos_adapter import CosmosAdapter

cosmos = CosmosAdapter.from_env()
retriever = HybridRetriever(LOCAL_GOLD, LOCAL_SILVER, mode=MODE, cosmos_adapter=cosmos)

STRATEGY_MAP = {
    "vector": RetrievalStrategy.VECTOR,
    "pathrag": RetrievalStrategy.PATHRAG,
    "graphrag": RetrievalStrategy.GRAPHRAG,
    "hybrid": RetrievalStrategy.HYBRID,
    "react": RetrievalStrategy.REACT,
}

print(f"Retriever initialized (mode={MODE}, cosmos={cosmos.is_configured})")

**Retrieve answers for all questions × strategies**

In [ ]:
import json
import time

all_results = []

for qi, qa in enumerate(EXPERT_QA):
    for strategy_name in STRATEGIES_TO_EVAL:
        strategy_enum = STRATEGY_MAP[strategy_name]
        print(f"[Q{qi+1}/{len(EXPERT_QA)}] [{strategy_name}] {qa['question'][:50]}...")

        start = time.time()
        result = retriever.retrieve(qa["question"], strategy_enum)
        elapsed = time.time() - start

        contexts = [c.get("text", c.get("text_english", "")) for c in result.chunks]

        all_results.append({
            "question": qa["question"],
            "ground_truth": qa["ground_truth"],
            "strategy": strategy_name,
            "answer": result.answer,
            "contexts": contexts,
            "num_chunks": len(result.chunks),
            "execution_time": round(elapsed, 2),
            "confidence": result.confidence,
        })
        print(f"  {len(result.chunks)} chunks, {elapsed:.1f}s — {result.answer[:80]}...")

print(f"\nRetrieved {len(all_results)} results ({len(EXPERT_QA)} questions × {len(STRATEGIES_TO_EVAL)} strategies)")

**RAGAS scoring**

In [ ]:
import asyncio
import os
import warnings
import nest_asyncio
nest_asyncio.apply()

warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

from openai import AsyncAzureOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas import SingleTurnSample
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, AnswerCorrectness

azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_key = os.environ["AZURE_OPENAI_API_KEY"]
azure_deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
azure_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
emb_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")

import httpx
client = AsyncAzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_key,
    api_version=azure_version,
    timeout=httpx.Timeout(120.0, connect=10.0),
    max_retries=3,
)

llm = llm_factory(model=azure_deployment, provider="openai", client=client,max_tokens=8192)
ragas_embeddings = embedding_factory(provider="openai", model=emb_deployment, client=client)

if not hasattr(ragas_embeddings, 'embed_query'):
    ragas_embeddings.embed_query = ragas_embeddings.embed_text
if not hasattr(ragas_embeddings, 'embed_documents'):
    ragas_embeddings.embed_documents = ragas_embeddings.embed_texts

scorers = {
    "faithfulness": Faithfulness(llm=llm),
    "answer_relevancy": AnswerRelevancy(llm=llm, embeddings=ragas_embeddings),
    "context_precision": ContextPrecision(llm=llm),
    "context_recall": ContextRecall(llm=llm),
    "answer_correctness": AnswerCorrectness(llm=llm, embeddings=ragas_embeddings),
}

metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

async def score_one(question, answer, contexts, ground_truth):
    ctx = contexts if contexts else ["No context retrieved."]
    scores = {}
    for mname in metric_names:
        try:
            scorer = scorers[mname]
            if mname == "faithfulness":
                r = await scorer.ascore(user_input=question, response=answer, retrieved_contexts=ctx)
            elif mname == "answer_relevancy":
                r = await scorer.ascore(user_input=question, response=answer)
            elif mname == "context_precision":
                r = await scorer.ascore(user_input=question, retrieved_contexts=ctx, reference=ground_truth)
            elif mname == "context_recall":
                r = await scorer.ascore(user_input=question, retrieved_contexts=ctx, reference=ground_truth)
            elif mname == "answer_correctness":
                r = await scorer.ascore(user_input=question, response=answer, reference=ground_truth)
            else:
                r = await scorer.ascore(user_input=question, response=answer, retrieved_contexts=ctx, reference=ground_truth)
            scores[mname] = round(float(r), 4)
        except Exception as e:
            print(f"    {mname} failed: {e}")
            scores[mname] = None
    return scores

# Score all results
ragas_scores = {}
strategy_names = sorted(set(r["strategy"] for r in all_results))

for sn in strategy_names:
    strategy_results = [r for r in all_results if r["strategy"] == sn]
    print(f"\nScoring {sn} ({len(strategy_results)} questions)...")

    per_q_scores = []
    for r in strategy_results:
        ctx = [str(c) for c in r.get("contexts", []) if c]
        if not ctx:
            ctx = ["No context retrieved."]
        scores = asyncio.get_event_loop().run_until_complete(
            score_one(r["question"], r["answer"], ctx, r["ground_truth"])
        )
        per_q_scores.append(scores)
        print(f"  Q: {r['question'][:40]}... → {scores}")

    avg = {}
    for m in metric_names:
        vals = [s[m] for s in per_q_scores if s.get(m) is not None]
        avg[m] = round(sum(vals) / len(vals), 4) if vals else None
    avg["per_question"] = per_q_scores
    avg["num_questions"] = len(strategy_results)
    avg["avg_execution_time"] = round(sum(r["execution_time"] for r in strategy_results) / len(strategy_results), 2)
    avg["avg_chunks"] = round(sum(r["num_chunks"] for r in strategy_results) / len(strategy_results), 1)
    ragas_scores[sn] = avg

print("\nScoring complete.")

**Print results table + save**

In [ ]:
print(f"\n{'='*90}")
print(f"RAGAS EVALUATION — {RUN_NAME}")
print(f"{'='*90}")
print(f"{'Strategy':<12} {'Faithful':>10} {'Relevancy':>10} {'Precision':>10} {'Recall':>10} {'Correct':>10} {'Time(s)':>8} {'Chunks':>7}")
print(f"{'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*8} {'-'*7}")

for sn in strategy_names:
    s = ragas_scores[sn]
    f_val = f"{s['faithfulness']:.4f}" if s.get('faithfulness') is not None else "--"
    r_val = f"{s['answer_relevancy']:.4f}" if s.get('answer_relevancy') is not None else "--"
    p_val = f"{s['context_precision']:.4f}" if s.get('context_precision') is not None else "--"
    c_val = f"{s['context_recall']:.4f}" if s.get('context_recall') is not None else "--"
    a_val = f"{s['answer_correctness']:.4f}" if s.get('answer_correctness') is not None else "--"
    print(f"{sn:<12} {f_val:>10} {r_val:>10} {p_val:>10} {c_val:>10} {a_val:>10} {s.get('avg_execution_time','--'):>8} {s.get('avg_chunks','--'):>7}")

print(f"{'='*90}")

# Save results
output = {
    "run_name": RUN_NAME,
    "timestamp": datetime.now().isoformat(),
    "mode": MODE,
    "num_questions": len(EXPERT_QA),
    "strategies_evaluated": STRATEGIES_TO_EVAL,
    "ragas_scores": ragas_scores,
    "detailed_results": all_results,
    "expert_qa": EXPERT_QA,
}

# Save locally
local_output = f"/tmp/pipeline/{OUTPUT_FILE}"
with open(local_output, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nSaved locally: {local_output}")

# Save to ADLS
adapter.write_bytes(ADLS_OUTPUT_PATH, json.dumps(output, indent=2, default=str).encode())
print(f"Saved to ADLS: {ADLS_OUTPUT_PATH}")

# List all previous evaluations on ADLS
eval_files = adapter.list_files("evaluation", "*.json")
print(f"\nAll evaluations on ADLS ({len(eval_files)}):")
for ef in eval_files:
    print(f"  {ef}")